[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/validation/VIX_STACKING_WF.ipynb)


# VIX Stacking WF v1 — le stacking (VIX_ML3 Stage 2) tient-il en walk-forward ?

**Contexte.** Le dernier run de `VIX_ML3.ipynb` (pipeline historique, en pause) a produit un
panel de 7436 modèles de base puis des méta-modèles Stage 2 entraînés sur leurs prédictions
OOF/test. Résultat sur ce run : le meilleur méta-modèle (**LightGBM_4cls, F1_dir=0.6578**)
dépasse le meilleur modèle individuel du panel (0.6473, CALM h5 GradientBoosting). Mais ce
chiffre vient du protocole propre à VIX_ML3 — un split OOF/test **statique**, pas de la
discipline walk-forward établie ensuite dans le reste du projet. Or le constat central de ce
projet (répété à chaque notebook de validation) est qu'un bon résultat en split statique ne
garantit rien en walk-forward — et le README indique explicitement que "le stacking a
systématiquement sous-performé le meilleur modèle individuel" dans les runs walk-forward
précédents. Ce notebook tranche : **le stacking tient-il, cette fois, en walk-forward ?**

**Protocole** (mêmes règles que `VIX_CHAMPION_WF`/`VIX_TFT_WF` : le champion candidat ET le
comparateur de référence sont recalculés **dans le même run**, pas juste cités) :
- Walk-forward expanding window, 5 folds (mêmes coupures que le reste du projet).
- Régime GLOBAL, tous horizons (1/2/3/5/7/10j) — le panel complet (7436 modèles, 4 régimes ×
  ~90 variantes par cellule) n'est pas reproduit ; on utilise un panel réduit mais
  représentatif de **5 algos** (XGBoost, LightGBM, RandomForest, GradientBoosting, CatBoost),
  cohérent avec le reste du projet, sur les N=8 features SHAP de référence.
- **Split interne causal** à l'intérieur de chaque train de fold : les derniers 25% du train
  servent de validation interne pour générer les méta-features OOF (au lieu du CV interne de
  VIX_ML3) — les 5 algos sont entraînés sur le reste du train, prédisent (probabilités) sur
  cette tranche de validation, et c'est sur ces prédictions OOF que le méta-modèle
  (LightGBM 4 classes, comme le meilleur Stage 2 du rapport) est entraîné. Les 5 algos sont
  ensuite ré-entraînés sur 100% du train pour produire les méta-features du vrai test.
- **STACKING** (panel + méta-modèle) vs **BASELINE** (RandomForest seul, même N=8, SMOTE —
  la config de référence établie, F1_dir≈0.610±0.025) : mêmes folds, mêmes features, seule
  la présence du stacking diffère.

**Verdict attendu.** Si STACKING ≥ BASELINE en walk-forward → le stacking devient une piste
sérieuse à généraliser. S'il s'effondre (comme les runs walk-forward précédents du README le
suggèrent) → le gain Stage 2 observé sur le run statique est requalifié en artefact de split,
comme cela a déjà été le cas pour le champion STRESS GradientBoosting et pour TFT.


In [1]:
import subprocess, sys
pkgs = ['xgboost', 'lightgbm', 'catboost', 'shap', 'xlsxwriter', 'imbalanced-learn', 'pyarrow']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


Installation OK


In [2]:
import os, time, json, warnings, random
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import shap
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from imblearn.over_sampling import SMOTE

SEED = 42; random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_NAME = 'VIX_STACKING_WF'
NOTEBOOK_VERSION = 'v1'

CONFIG = {
    'flat_thr': 0.003,
    'horizons': [1, 2, 3, 5, 7, 10],
    'N': 8,                     # config de référence établie
    'base_algos': ['XGBoost', 'LightGBM', 'RandomForest', 'GradientBoosting', 'CatBoost'],
    'meta_algo': 'LightGBM',    # meilleur Stage 2 du rapport VIX_ML3 (F1_dir=0.6578)
    'inner_val_frac': 0.25,     # dernier quart du train -> validation interne (méta-features OOF)
    'n_wf_folds': 5,
    'min_train_frac': 0.40,     # doit matcher VIX_FINAL_FEATURES
    'shap_sample': 500,
    'pool_prefilter': 450,
    'min_train_rows': 150, 'min_test_rows': 20,
}
TARGET_COL = 'VIX_Amplitude_Class'
RESULTS_CSV = 'vix_stacking_wf_results.csv'
GITHUB_REPO = 'LP-D/claude'
FEATURES_BRANCH = 'results/vix-final-features'
RESULTS_BRANCH = 'results/vix-stacking-wf'

BASELINE_F1_DIR = 0.610  # référence README (GLOBAL RandomForest h=5j N=8 SHAP SMOTE)

n_combos = len(CONFIG['horizons']) * CONFIG['n_wf_folds'] * 2  # x2 : STACKING + BASELINE
print(f"{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | horizons={CONFIG['horizons']} | "
      f"algos panel={CONFIG['base_algos']} | méta={CONFIG['meta_algo']} | "
      f"grille = {n_combos} lignes ({len(CONFIG['horizons'])}h × {CONFIG['n_wf_folds']}folds × 2 configs)")


VIX_STACKING_WF v1 | horizons=[1, 2, 3, 5, 7, 10] | algos panel=['XGBoost', 'LightGBM', 'RandomForest', 'GradientBoosting', 'CatBoost'] | méta=LightGBM | grille = 60 lignes (6h × 5folds × 2 configs)


In [3]:
# ============================================================
# CHARGEMENT DU DATASET PARTAGÉ (produit par VIX_FINAL_FEATURES)
# ============================================================
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

if not os.path.exists('vix_final_features.parquet'):
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    url = f"https://{auth}github.com/{GITHUB_REPO}.git"
    workdir = "/content/_vix_features_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", FEATURES_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode != 0:
        raise RuntimeError(
            "Impossible de récupérer le dataset partagé depuis "
            f"'{FEATURES_BRANCH}'. As-tu bien exécuté VIX_FINAL_FEATURES.ipynb en premier "
            f"(et poussé son résultat) ? Détail: {clone.stderr[-500:]}")
    subprocess.run(["cp", f"{workdir}/vix_final_features.parquet", "."], check=True)
    subprocess.run(["cp", f"{workdir}/vix_final_features_meta.json", "."], check=True)
    print(f"[PULL OK] Dataset récupéré depuis '{FEATURES_BRANCH}'")
else:
    print("[SKIP] vix_final_features.parquet déjà présent localement")

df_features = pd.read_parquet('vix_final_features.parquet')
with open('vix_final_features_meta.json') as f:
    meta = json.load(f)
FEATURE_POOL = meta['feature_pool']
VIX_COL = meta['vix_col']; SPX_COL = meta['spx_col']
print(f"Dataset: {df_features.shape} | VIX={VIX_COL} | pool: {len(FEATURE_POOL)} features "
      f"(dont {len(meta['interaction_features'])} interactions) | source: {meta['date_min']} → {meta['date_max']}")

all_dates = df_features.dropna(how='all').index.sort_values()
n_obs = len(all_dates)
first_cut = int(n_obs * CONFIG['min_train_frac'])
test_span = (n_obs - first_cut) // CONFIG['n_wf_folds']
FOLD_CUTS = [first_cut + k * test_span for k in range(CONFIG['n_wf_folds'] + 1)]
FOLD_CUTS[-1] = n_obs
for k in range(CONFIG['n_wf_folds']):
    print(f"  Fold {k+1}: train → {all_dates[FOLD_CUTS[k]-1].date()} | "
          f"test {all_dates[FOLD_CUTS[k]].date()} → {all_dates[FOLD_CUTS[k+1]-1].date()}")


[PULL OK] Dataset récupéré depuis 'results/vix-final-features'
Dataset: (6908, 1300) | VIX=IDX_VIX | pool: 1102 features (dont 21 interactions) | source: 2000-01-03 → 2026-07-21
  Fold 1: train → 2010-08-16 | test 2010-08-17 → 2013-10-21
  Fold 2: train → 2013-10-21 | test 2013-10-22 → 2016-12-28
  Fold 3: train → 2016-12-28 | test 2016-12-29 → 2020-03-06
  Fold 4: train → 2020-03-06 | test 2020-03-09 → 2023-05-12
  Fold 5: train → 2023-05-12 | test 2023-05-15 → 2026-07-21


In [4]:
def build_target(vix_series, horizon, split_idx):
    vix = vix_series.ffill().bfill(); vix_tr = vix.iloc[:split_idx]
    calm_thr = vix_tr.quantile(0.33); stress_thr = vix_tr.quantile(0.67)
    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr] = 'CALM'; regime[vix >= stress_thr] = 'STRESS'
    ret = (vix.shift(-horizon) / vix) - 1
    flat = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat].dropna(); reg_r = regime.reindex(ret.index)
    cut_date = vix.index[min(split_idx, len(vix) - 1)]
    ret_tr = ret.loc[ret.index < cut_date]; reg_tr = reg_r.loc[ret_tr.index]
    thr = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_tr[reg_tr == reg]
        thr[reg] = (sub.quantile(0.25) if len(sub) >= 20 else ret_tr.quantile(0.25),
                    sub.quantile(0.75) if len(sub) >= 20 else ret_tr.quantile(0.75))
    thr['GLOBAL'] = (ret_tr.quantile(0.25), ret_tr.quantile(0.75))
    def classify(r, reg):
        q25, q75 = thr.get(reg, (0, 0))
        if r < q25: return 0
        if r < 0:   return 1
        if r < q75: return 2
        return 3
    target = pd.Series([classify(r, reg_r[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    return target, reg_r, thr

def metrics(y_true, y_pred):
    dm = {0: 'DOWN', 1: 'DOWN', 2: 'UP', 3: 'UP'}
    yd_t = [dm[y] for y in y_true]; yd_p = [dm[y] for y in y_pred]
    m = {'F1_4cls': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
         'Acc_dir': round(accuracy_score(yd_t, yd_p), 4),
         'F1_dir': round(f1_score(yd_t, yd_p, average='macro', zero_division=0), 4)}
    ui = [i for i, y in enumerate(y_true) if dm[y] == 'UP']
    di = [i for i, y in enumerate(y_true) if dm[y] == 'DOWN']
    if len(ui) >= 10:
        yt = ['FORT' if y_true[i] == 3 else 'FAIBLE' for i in ui]
        yp = ['FORT' if y_pred[i] == 3 else 'FAIBLE' for i in ui]
        m['F1_UP_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_UP_FORT'] = np.nan
    if len(di) >= 10:
        yt = ['FORT' if y_true[i] == 0 else 'FAIBLE' for i in di]
        yp = ['FORT' if y_pred[i] == 0 else 'FAIBLE' for i in di]
        m['F1_DOWN_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_DOWN_FORT'] = np.nan
    return m

def get_clf(algo):
    if algo == 'XGBoost':
        return XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=3, eval_metric='mlogloss',
                             objective='multi:softprob', random_state=SEED, n_jobs=-1, verbosity=0)
    if algo == 'LightGBM':
        return LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, num_leaves=31,
                              min_child_samples=10, subsample=0.8, class_weight='balanced',
                              random_state=SEED, verbose=-1, n_jobs=-1)
    if algo == 'RandomForest':
        return RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5,
                                      class_weight='balanced', random_state=SEED, n_jobs=-1)
    if algo == 'GradientBoosting':
        return GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                          min_samples_leaf=10, subsample=0.8, random_state=SEED)
    if algo == 'CatBoost':
        return CatBoostClassifier(iterations=200, depth=6, learning_rate=0.05, loss_function='MultiClass',
                                  auto_class_weights='Balanced', random_state=SEED, verbose=False,
                                  allow_writing_files=False)
    raise ValueError(algo)

def get_samp():
    return SMOTE(random_state=SEED)

def shap_rank(X_tr, y_tr, pool_names, top_n, prefilter):
    nf = X_tr.shape[1]
    if nf > prefilter:
        pf = XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                           eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
        pf.fit(X_tr, y_tr); keep = np.argsort(pf.feature_importances_)[::-1][:prefilter]
    else:
        keep = np.arange(nf)
    Xk = X_tr[:, keep]
    pilot = XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                          eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
    pilot.fit(Xk, y_tr)
    sv = np.abs(np.array(shap.TreeExplainer(pilot).shap_values(Xk[:min(CONFIG['shap_sample'], len(Xk))])))
    nfk = Xk.shape[1]
    feat_axes = [ax for ax in range(sv.ndim) if sv.shape[ax] == nfk]
    if len(feat_axes) == 1:
        arr = sv.mean(axis=tuple(ax for ax in range(sv.ndim) if ax != feat_axes[0]))
    else:
        arr = np.asarray(pilot.feature_importances_)
    order = np.argsort(np.asarray(arr).ravel())[::-1][:top_n]
    return list(keep[order])

def proba_aligned(clf, X, n_classes=4):
    """predict_proba réaligné sur des colonnes [0..n_classes-1] fixes, même si une
    classe est absente de l'échantillon d'entraînement (SMOTE peut échouer/tomber en
    repli si une classe manque totalement, auquel cas predict_proba a moins de
    colonnes que n_classes) -- sans ce réalignement, l'ordre des colonnes de
    probabilité varierait d'un algo/fold à l'autre et casserait le hstack des
    méta-features."""
    p = clf.predict_proba(X)
    if p.shape[1] == n_classes:
        return p
    full = np.zeros((p.shape[0], n_classes))
    full[:, clf.classes_] = p
    return full

print("Helpers OK (build_target, metrics, get_clf, get_samp, shap_rank, proba_aligned)")


Helpers OK (build_target, metrics, get_clf, get_samp, shap_rank, proba_aligned)


In [5]:
# ============================================================
# SYNCHRONISATION DE LA PROGRESSION (checkpoint résumable — même principe
# que les autres notebooks à grille : le disque Colab ne survit pas à une
# déconnexion, on republie régulièrement pendant le run, pas seulement à la fin).
# ============================================================
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

_PUSH_WORKDIR = "/content/_vix_stacking_wf_push"
KEY_COLS = ['horizon', 'fold', 'config']

def pull_progress():
    if os.path.exists(RESULTS_CSV):
        print(f"[SKIP PULL] {RESULTS_CSV} déjà présent localement.")
        return
    if not GITHUB_TOKEN:
        print("[SKIP PULL] Pas de GITHUB_TOKEN — démarrage de zéro dans ce runtime.")
        return
    url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    exists = subprocess.run(["git", "ls-remote", "--exit-code", "--heads", url, RESULTS_BRANCH],
                            capture_output=True, text=True)
    if exists.returncode != 0:
        print(f"[INFO] Aucune progression antérieure sur '{RESULTS_BRANCH}' — nouveau run.")
        return
    workdir = "/content/_vix_stacking_wf_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", RESULTS_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode == 0 and os.path.exists(f"{workdir}/{RESULTS_CSV}"):
        subprocess.run(["cp", f"{workdir}/{RESULTS_CSV}", "."], check=True)
        n = sum(1 for _ in open(RESULTS_CSV)) - 1
        print(f"[PULL OK] Progression antérieure récupérée : {n} lignes déjà faites.")
    else:
        print(f"[WARN] Branche '{RESULTS_BRANCH}' trouvée mais {RESULTS_CSV} absent — nouveau run.")

def push_progress(label=''):
    if not GITHUB_TOKEN or not os.path.exists(RESULTS_CSV):
        return False
    try:
        subprocess.run(["rm", "-rf", _PUSH_WORKDIR], check=False)
        url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
        clone = subprocess.run(["git", "clone", url, _PUSH_WORKDIR], capture_output=True, text=True)
        if clone.returncode != 0: return False
        exists = subprocess.run(["git", "-C", _PUSH_WORKDIR, "ls-remote", "--exit-code", "--heads",
                                  "origin", RESULTS_BRANCH], capture_output=True, text=True)
        if exists.returncode == 0:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH,
                             f"origin/{RESULTS_BRANCH}"], check=True)
        else:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH], check=True)
        subprocess.run(["cp", RESULTS_CSV, f"{_PUSH_WORKDIR}/{RESULTS_CSV}"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.email", "vix-colab@users.noreply.github.com"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.name", "VIX Stacking WF Colab run"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", RESULTS_CSV], check=True)
        commit = subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                                 f"Progression Stacking WF {label} — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                                capture_output=True, text=True)
        if 'nothing to commit' in (commit.stdout or ''):
            return True
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        ok = push.returncode == 0
        if ok: print(f"  [CHECKPOINT PUSHÉ] {label} ({pd.Timestamp.now():%H:%M:%S})")
        else: print(f"  [WARN push checkpoint] {push.stderr[-300:]}")
        return ok
    except Exception as e:
        print(f"  [WARN push checkpoint] {e}")
        return False

pull_progress()

done_keys = set()
if os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0:
    prev = pd.read_csv(RESULTS_CSV, usecols=KEY_COLS)
    done_keys = set(map(tuple, prev.values.tolist()))
    print(f"[REPRISE] {len(done_keys)}/{n_combos} lignes déjà faites — reprise en cours.")
else:
    print("[DÉMARRAGE] Aucun résultat existant — nouveau run.")

def already_done(h, fold, config):
    return (h, fold, config) in done_keys

def save_row(row):
    header = not (os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0)
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=header, index=False)
    done_keys.add(tuple(row[c] for c in KEY_COLS))


[INFO] Aucune progression antérieure sur 'results/vix-stacking-wf' — nouveau run.
[DÉMARRAGE] Aucun résultat existant — nouveau run.


### Stacking à 2 étages — méthodologie causale utilisée ici

Un ensemble de stacking entraîne un **méta-modèle** sur les prédictions (ici des
probabilités par classe) d'un panel de modèles de base, plutôt que de les combiner par un
vote simple. Le piège classique : si le méta-modèle est entraîné sur les prédictions des
modèles de base **sur les données qui ont servi à les entraîner**, il apprend à corriger le
sur-ajustement de ces modèles plutôt qu'un vrai signal — d'où la nécessité de prédictions
**out-of-fold (OOF)**, jamais vues par le modèle de base qui les produit.

Ici, la séparation OOF est obtenue par un **split causal interne** au train de chaque fold
walk-forward : les derniers `inner_val_frac` (25%) du train servent de validation interne
(jamais utilisés pour entraîner les modèles de base qui produisent leurs propres prédictions
OOF), le reste sert à l'entraînement de ces modèles de base. Le méta-modèle apprend ensuite
la relation entre ces probabilités OOF et les vraies classes. Pour la prédiction finale sur le
test du fold, les modèles de base sont **ré-entraînés sur 100% du train** (ils disposent alors
de plus de données que lors de la génération des méta-features OOF, ce qui est cohérent avec
la pratique standard de stacking) et leurs probabilités sur le test alimentent le méta-modèle
déjà entraîné.


In [6]:
# ============================================================
# MOTEUR : par (horizon, fold), calcule STACKING (panel 5 algos + méta-modèle,
# split interne causal) et BASELINE (RandomForest seul), refaits dans le même
# run pour une comparaison directe et honnête.
# ============================================================
def build_meta_features(probas_dict):
    return np.hstack([probas_dict[a] for a in CONFIG['base_algos']])

t0 = time.time()
n_done_session = 0

for h in CONFIG['horizons']:
    for k in range(CONFIG['n_wf_folds']):
        cut, nxt = FOLD_CUTS[k], FOLD_CUTS[k + 1]
        cut_date, nxt_date = all_dates[cut], all_dates[nxt - 1]

        if already_done(h, k + 1, 'STACKING') and already_done(h, k + 1, 'BASELINE'):
            continue

        target, reg_r, _ = build_target(df_features[VIX_COL], h, cut)
        idx = target.index
        tr_mask = np.asarray(idx < cut_date)
        te_mask = np.asarray((idx >= cut_date) & (idx <= nxt_date))
        y_tr_full = target.values[tr_mask].astype(int)
        y_te = target.values[te_mask].astype(int)
        if len(y_tr_full) < CONFIG['min_train_rows'] or len(y_te) < CONFIG['min_test_rows']:
            print(f"h={h}j fold{k+1}: échantillon insuffisant (tr={len(y_tr_full)} te={len(y_te)}) — sauté")
            continue

        X_pool = df_features[FEATURE_POOL].reindex(idx)
        sc = RobustScaler()
        X_tr_full = sc.fit_transform(np.nan_to_num(X_pool.values[tr_mask]))
        X_te = sc.transform(np.nan_to_num(X_pool.values[te_mask]))

        cols = shap_rank(X_tr_full, y_tr_full, FEATURE_POOL, CONFIG['N'], CONFIG['pool_prefilter'])
        X_tr_full_n, X_te_n = X_tr_full[:, cols], X_te[:, cols]

        n_tr = len(y_tr_full)
        inner_cut = int(n_tr * (1 - CONFIG['inner_val_frac']))
        X_inner_tr, X_inner_val = X_tr_full_n[:inner_cut], X_tr_full_n[inner_cut:]
        y_inner_tr, y_inner_val = y_tr_full[:inner_cut], y_tr_full[inner_cut:]

        row_common = {'horizon': h, 'fold': k + 1, 'n_train': len(y_tr_full), 'n_test': len(y_te),
                     'test_start': str(cut_date.date()), 'test_end': str(nxt_date.date())}

        # ---- BASELINE : RandomForest seul, refit sur tout le train (config établie) ----
        if not already_done(h, k + 1, 'BASELINE'):
            try:
                Xr, yr = get_samp().fit_resample(X_tr_full_n, y_tr_full)
            except Exception:
                Xr, yr = X_tr_full_n, y_tr_full
            base_rf = get_clf('RandomForest'); base_rf.fit(Xr, yr)
            met_b = metrics(y_te, base_rf.predict(X_te_n))
            save_row({**row_common, 'config': 'BASELINE', **met_b})
            print(f"h={h:2d}j fold{k+1} BASELINE : F1_dir={met_b['F1_dir']:.3f}")

        # ---- STACKING : panel de 5 algos (split interne causal) -> méta-modèle ----
        if not (len(y_inner_tr) >= CONFIG['min_train_rows'] and len(y_inner_val) >= CONFIG['min_test_rows']):
            print(f"h={h}j fold{k+1}: split interne insuffisant (inner_tr={len(y_inner_tr)} "
                  f"inner_val={len(y_inner_val)}) — STACKING sauté")
        elif not already_done(h, k + 1, 'STACKING'):
            oof_probas, test_probas = {}, {}
            for algo in CONFIG['base_algos']:
                try:
                    Xr, yr = get_samp().fit_resample(X_inner_tr, y_inner_tr)
                except Exception:
                    Xr, yr = X_inner_tr, y_inner_tr
                clf_inner = get_clf(algo); clf_inner.fit(Xr, yr)
                oof_probas[algo] = proba_aligned(clf_inner, X_inner_val)

                try:
                    Xr2, yr2 = get_samp().fit_resample(X_tr_full_n, y_tr_full)
                except Exception:
                    Xr2, yr2 = X_tr_full_n, y_tr_full
                clf_full = get_clf(algo); clf_full.fit(Xr2, yr2)
                test_probas[algo] = proba_aligned(clf_full, X_te_n)

            X_meta_tr = build_meta_features(oof_probas)
            X_meta_te = build_meta_features(test_probas)
            meta_clf = get_clf(CONFIG['meta_algo'])
            meta_clf.fit(X_meta_tr, y_inner_val)
            met_s = metrics(y_te, meta_clf.predict(X_meta_te))
            save_row({**row_common, 'config': 'STACKING', **met_s})
            print(f"h={h:2d}j fold{k+1} STACKING  : F1_dir={met_s['F1_dir']:.3f} "
                  f"F1_UP_FORT={met_s['F1_UP_FORT']} F1_DOWN_FORT={met_s['F1_DOWN_FORT']}")

        n_done_session += 1
        if n_done_session % 5 == 0:
            push_progress(label=f"{len(done_keys)}/{n_combos}")

push_progress(label=f"fin de session ({len(done_keys)}/{n_combos})")
print(f"\n[TERMINÉ] {len(done_keys)}/{n_combos} lignes en {(time.time()-t0)/60:.1f}min")


h= 1j fold1 BASELINE : F1_dir=0.556
h= 1j fold1 STACKING  : F1_dir=0.480 F1_UP_FORT=0.315 F1_DOWN_FORT=0.3205
h= 1j fold2 BASELINE : F1_dir=0.552
h= 1j fold2 STACKING  : F1_dir=0.567 F1_UP_FORT=0.4316 F1_DOWN_FORT=0.4623
h= 1j fold3 BASELINE : F1_dir=0.582
h= 1j fold3 STACKING  : F1_dir=0.531 F1_UP_FORT=0.4082 F1_DOWN_FORT=0.5051
h= 1j fold4 BASELINE : F1_dir=0.579
h= 1j fold4 STACKING  : F1_dir=0.528 F1_UP_FORT=0.3893 F1_DOWN_FORT=0.4505
h= 1j fold5 BASELINE : F1_dir=0.586
h= 1j fold5 STACKING  : F1_dir=0.537 F1_UP_FORT=0.3971 F1_DOWN_FORT=0.4703
  [CHECKPOINT PUSHÉ] 10/60 (07:47:53)
h= 2j fold1 BASELINE : F1_dir=0.573
h= 2j fold1 STACKING  : F1_dir=0.532 F1_UP_FORT=0.4268 F1_DOWN_FORT=0.3879
h= 2j fold2 BASELINE : F1_dir=0.563
h= 2j fold2 STACKING  : F1_dir=0.497 F1_UP_FORT=0.3841 F1_DOWN_FORT=0.4444
h= 2j fold3 BASELINE : F1_dir=0.586
h= 2j fold3 STACKING  : F1_dir=0.548 F1_UP_FORT=0.3976 F1_DOWN_FORT=0.5173
h= 2j fold4 BASELINE : F1_dir=0.566
h= 2j fold4 STACKING  : F1_dir=0.555 F1

In [7]:
# ============================================================
# SYNTHÈSE : STACKING vs BASELINE en walk-forward
# ============================================================
df_swf = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
print(f"Progression: {len(df_swf)}/{n_combos} ({len(df_swf)/max(n_combos,1):.1%})")

if len(df_swf):
    piv = df_swf.pivot_table(index=['horizon', 'fold'], columns='config',
                             values=['F1_dir', 'F1_UP_FORT', 'F1_DOWN_FORT'])
    if ('F1_dir', 'STACKING') in piv.columns and ('F1_dir', 'BASELINE') in piv.columns:
        delta = pd.DataFrame({
            'F1_dir_BASELINE': piv[('F1_dir', 'BASELINE')].round(4),
            'F1_dir_STACKING': piv[('F1_dir', 'STACKING')].round(4),
            'delta_F1_dir': (piv[('F1_dir', 'STACKING')] - piv[('F1_dir', 'BASELINE')]).round(4),
            'F1_UP_FORT_BASELINE': piv[('F1_UP_FORT', 'BASELINE')].round(4),
            'F1_UP_FORT_STACKING': piv[('F1_UP_FORT', 'STACKING')].round(4),
            'F1_DOWN_FORT_BASELINE': piv[('F1_DOWN_FORT', 'BASELINE')].round(4),
            'F1_DOWN_FORT_STACKING': piv[('F1_DOWN_FORT', 'STACKING')].round(4),
        }).dropna(subset=['F1_dir_BASELINE', 'F1_dir_STACKING']).reset_index()
        print("\n### STACKING vs BASELINE par (horizon, fold) ###")
        print(delta.to_string(index=False))

        by_h = delta.groupby('horizon')[['F1_dir_BASELINE', 'F1_dir_STACKING', 'delta_F1_dir']].mean().round(4)
        print("\n### Moyenne par horizon ###")
        print(by_h.to_string())

        mean_baseline = delta['F1_dir_BASELINE'].mean()
        mean_stacking = delta['F1_dir_STACKING'].mean()
        mean_delta = delta['delta_F1_dir'].mean()
        win_rate = (delta['delta_F1_dir'] > 0).mean()
        print(f"\nF1_dir moyen — BASELINE(recalculé ici)={mean_baseline:.4f} "
              f"(référence README: {BASELINE_F1_DIR}) | STACKING={mean_stacking:.4f} | "
              f"delta moyen={mean_delta:+.4f} | STACKING gagne sur {win_rate:.0%} des "
              f"(horizon, fold)")

        if mean_delta >= 0.01:
            print("\n[VERDICT] Le stacking bat la référence GLOBAL RandomForest en walk-forward "
                  "strict (delta positif et au-dessus du bruit) — résultat qui contredit le "
                  "constat historique du projet. À creuser sérieusement avant de généraliser "
                  "(vérifier la stabilité sur d'autres seeds/regimes avant adoption).")
        elif mean_delta <= -0.01:
            print("\n[VERDICT] Le stacking sous-performe la référence en walk-forward (delta "
                  f"négatif, {mean_delta:+.4f}) — cohérent avec le constat historique du projet "
                  "et avec la règle centrale : un gain en split statique (VIX_ML3, +0.0105 "
                  "Stage 2 vs meilleur modèle) ne survit pas à une validation walk-forward "
                  "honnête. Le gain observé dans le rapport VIX_ML3 est requalifié en artefact "
                  "de split, comme pour le champion STRESS GradientBoosting et pour TFT.")
        else:
            print(f"\n[VERDICT] Delta dans le bruit ({mean_delta:+.4f}, <0.01 en valeur absolue) "
                  "— le stacking n'apporte ni gain ni perte significative par rapport à la "
                  "référence ; pas de raison de complexifier le pipeline pour ce gain nul.")

        try:
            with pd.ExcelWriter('VIX_STACKING_WF_report.xlsx', engine='xlsxwriter') as w:
                delta.to_excel(w, 'Delta_par_fold', index=False)
                by_h.reset_index().to_excel(w, 'Par_horizon', index=False)
                df_swf.to_excel(w, 'Detail', index=False)
                pd.DataFrame([{'F1_dir_stacking_moyen': mean_stacking,
                                'F1_dir_baseline_recalcule': mean_baseline,
                                'F1_dir_reference_readme': BASELINE_F1_DIR,
                                'delta_moyen': mean_delta, 'taux_victoire_stacking': win_rate}]
                            ).to_excel(w, 'Meta', index=False)
            print("\n[SAVE] VIX_STACKING_WF_report.xlsx")
        except Exception as e:
            print(f"[WARN Export] {e}")
    else:
        print("Pas encore assez de lignes des deux configs (STACKING/BASELINE) pour comparer.")
else:
    print("Aucun résultat pour l'instant.")
print(f"\n[NOTE] {RESULTS_CSV} contient le détail complet — le recharger pour reprendre.")


Progression: 60/60 (100.0%)

### STACKING vs BASELINE par (horizon, fold) ###
 horizon  fold  F1_dir_BASELINE  F1_dir_STACKING  delta_F1_dir  F1_UP_FORT_BASELINE  F1_UP_FORT_STACKING  F1_DOWN_FORT_BASELINE  F1_DOWN_FORT_STACKING
       1     1           0.5557           0.4803       -0.0754               0.2800               0.3150                 0.5583                 0.3205
       1     2           0.5524           0.5674        0.0150               0.3061               0.4316                 0.6516                 0.4623
       1     3           0.5821           0.5313       -0.0508               0.2709               0.4082                 0.6624                 0.5051
       1     4           0.5788           0.5282       -0.0506               0.3493               0.3893                 0.5903                 0.4505
       1     5           0.5857           0.5370       -0.0487               0.2259               0.3971                 0.6402                 0.4703
       2     1  

In [8]:
# ============================================================
# PUSH FINAL DU RAPPORT (xlsx) EN PLUS DU CSV DE PROGRESSION
# ============================================================
def push_report_file():
    if not GITHUB_TOKEN or not os.path.exists('VIX_STACKING_WF_report.xlsx'):
        print("[SKIP] Pas de token ou pas de rapport à pousser.")
        return
    try:
        subprocess.run(["cp", "VIX_STACKING_WF_report.xlsx", f"{_PUSH_WORKDIR}/VIX_STACKING_WF_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", "VIX_STACKING_WF_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Rapport Stacking WF agrégé — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        if push.returncode == 0:
            print(f"[PUSH OK] VIX_STACKING_WF_report.xlsx sur '{RESULTS_BRANCH}'")
        else:
            print(f"[WARN] {push.stderr[-300:]}")
    except Exception as e:
        print(f"[WARN] {e}")

push_progress(label='rapport final')
push_report_file()


[PUSH OK] VIX_STACKING_WF_report.xlsx sur 'results/vix-stacking-wf'
